# Document Question Answering System (RAG)

This notebook implements a simple **Retrieval-Augmented Generation (RAG)** pipeline that answers questions about a custom document.

Instead of relying only on a language model's internal (and possibly outdated or missing) knowledge, the system:
1. Splits a document into chunks
2. Embeds each chunk into a vector space
3. Stores the vectors in a vector database (FAISS)
4. Retrieves the most relevant chunks for a user's question
5. Feeds those chunks to a language model as context so it can generate a grounded answer

The stack used here is entirely free/local — **Sentence-Transformers** for embeddings, **FAISS** for similarity search, and **Flan-T5** (Hugging Face) for answer generation — so it runs without any API keys.

In [1]:
!pip install -q sentence-transformers faiss-cpu transformers torch
!pip install -q transformers torch sentencepiece


[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# **Import Libraries**

In [2]:
import os
import textwrap
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import pipeline

C:\Users\MAYANK CHAUHAN\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# **Create Sample Document**

RAG is meant for custom/private data. Here we generate a small notes document about Machine Learning and Deep Learning (a natural follow-up to the autoencoder project from Week 6) and save it as a `.txt` file so the pipeline can load it like any real document.

In [3]:
document_text = """
Machine Learning and Deep Learning: Study Notes

Introduction to Machine Learning
Machine learning (ML) is a branch of artificial intelligence in which systems learn patterns from data instead of
being explicitly programmed with rules. A model is trained on historical data, and it uses the patterns it finds to
make predictions or decisions on new, unseen data. The three broad categories of machine learning are supervised
learning, unsupervised learning, and reinforcement learning.

Supervised Learning
In supervised learning, the model is trained on labeled data, meaning every training example includes both an
input and the correct output. Common supervised tasks include classification, where the output is a category
(such as spam or not spam), and regression, where the output is a continuous number (such as predicting a house
price). Popular supervised algorithms include linear regression, decision trees, random forests, and neural
networks.

Unsupervised Learning
Unsupervised learning works with unlabeled data. The goal is to discover hidden structure in the data rather than
predict a known output. Clustering algorithms, such as k-means, group similar data points together. Dimensionality
reduction techniques, such as PCA, compress data into fewer dimensions while preserving as much information as
possible. Autoencoders, a type of neural network, can also be used for unsupervised representation learning.

Reinforcement Learning
Reinforcement learning trains an agent to make a sequence of decisions by rewarding good actions and penalizing bad
ones. The agent interacts with an environment, receives feedback in the form of rewards, and gradually learns a
policy that maximizes cumulative reward over time. It is widely used in robotics, game playing, and resource
management.

Neural Networks
A neural network is a computational model inspired by the structure of the human brain. It consists of layers of
interconnected nodes, called neurons, organized into an input layer, one or more hidden layers, and an output
layer. Each connection has a weight that is adjusted during training using an algorithm called backpropagation,
which minimizes a loss function through gradient descent.

Deep Learning
Deep learning refers to neural networks with many hidden layers. These deep architectures can automatically learn
increasingly abstract features from raw data, which is why they perform so well on complex tasks like image
recognition, speech recognition, and natural language processing. Training deep networks generally requires large
amounts of data and significant computing power, often provided by GPUs.

Autoencoders
An autoencoder is a type of neural network trained to reconstruct its own input. It consists of an encoder, which
compresses the input into a smaller latent representation, and a decoder, which reconstructs the original input
from that representation. Autoencoders are commonly used for dimensionality reduction, anomaly detection, and
denoising. A denoising autoencoder is specifically trained on noisy inputs paired with clean targets, so it learns
to remove noise while preserving the important structure of the data.

Convolutional Neural Networks
Convolutional Neural Networks, or CNNs, are a specialized type of neural network designed for grid-like data such
as images. They use convolutional layers with learnable filters that slide across the input to detect local
patterns like edges, textures, and shapes. Pooling layers reduce the spatial size of the data, which lowers
computation and helps the network become more robust to small shifts in the input. CNNs are the backbone of most
modern computer vision systems.

Recurrent Neural Networks
Recurrent Neural Networks, or RNNs, are designed to process sequential data, such as text or time series, by
maintaining a hidden state that carries information from previous time steps. Variants such as Long Short-Term
Memory (LSTM) networks and Gated Recurrent Units (GRUs) were developed to address the vanishing gradient problem,
allowing the network to learn long-range dependencies in sequences.

Transformers
The transformer architecture replaced recurrence with a mechanism called self-attention, which allows the model to
weigh the importance of every other token in a sequence when processing a given token. This makes transformers
highly parallelizable and effective at capturing long-range dependencies. Transformers are the foundation of
modern large language models, including the models used for text generation, translation, and question answering.

Retrieval-Augmented Generation
Retrieval-Augmented Generation, or RAG, combines a retrieval system with a generative language model. Instead of
relying only on knowledge baked into the model's parameters during training, a RAG system first retrieves relevant
passages from an external knowledge source, such as a document collection, and then passes those passages to the
language model as additional context. This grounds the model's answers in the retrieved information, which reduces
hallucination and allows the system to answer questions about private or up-to-date data that the model was never
trained on.

Applications of Deep Learning
Deep learning powers many real-world applications, including image classification, object detection, machine
translation, speech-to-text systems, recommendation engines, chatbots, and generative art. As models and datasets
continue to grow, deep learning systems are increasingly used in enterprise search, knowledge assistants, and
AI-powered documentation tools such as RAG-based question answering systems.
""".strip()

with open("sample_document.txt", "w", encoding="utf-8") as f:
    f.write(document_text)

print(f"Saved sample_document.txt ({len(document_text)} characters)")

Saved sample_document.txt (5622 characters)


# **Document Ingestion**

Load the raw document text. In a real project this is where you would load a PDF, resume, or research paper (e.g. with `pypdf` / `PyPDF2`) and convert it into plain text.

In [4]:
with open("sample_document.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Document length: {len(raw_text)} characters\n")
print("Preview:\n")
print(raw_text[:400] + "...")

Document length: 5622 characters

Preview:

Machine Learning and Deep Learning: Study Notes

Introduction to Machine Learning
Machine learning (ML) is a branch of artificial intelligence in which systems learn patterns from data instead of
being explicitly programmed with rules. A model is trained on historical data, and it uses the patterns it finds to
make predictions or decisions on new, unseen data. The three broad categories of machine...


# **Text Chunking**

The document is split into smaller, overlapping chunks. Overlap helps preserve context that would otherwise be cut in half at a chunk boundary.

In [5]:
def chunk_text(text, chunk_size=100, overlap=20):
    """Split text into overlapping chunks of `chunk_size` words."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


chunks = chunk_text(raw_text, chunk_size=100, overlap=20)

print(f"Number of chunks: {len(chunks)}\n")
print("Sample chunk (#0):\n")
print(textwrap.fill(chunks[0], width=100))

Number of chunks: 11

Sample chunk (#0):

Machine Learning and Deep Learning: Study Notes Introduction to Machine Learning Machine learning
(ML) is a branch of artificial intelligence in which systems learn patterns from data instead of
being explicitly programmed with rules. A model is trained on historical data, and it uses the
patterns it finds to make predictions or decisions on new, unseen data. The three broad categories
of machine learning are supervised learning, unsupervised learning, and reinforcement learning.
Supervised Learning In supervised learning, the model is trained on labeled data, meaning every
training example includes both an input and the correct output. Common supervised tasks include


# **Embedding Creation**

Each chunk is converted into a dense vector that captures its semantic meaning, using the `all-MiniLM-L6-v2` sentence-embedding model.

In [6]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(
    chunks,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print(f"Embeddings shape: {chunk_embeddings.shape}")

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.31it/s]

Embeddings shape: (11, 384)


# **Vector Database (FAISS)**

The chunk embeddings are stored in a FAISS index for fast similarity search. Since the embeddings are normalized, inner product search (`IndexFlatIP`) is equivalent to cosine similarity.

In [7]:
embedding_dim = chunk_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(np.array(chunk_embeddings, dtype="float32"))

print(f"FAISS index built with {index.ntotal} vectors of dimension {embedding_dim}")

FAISS index built with 11 vectors of dimension 384


# **Query Processing & Context Retrieval**

The user's question is embedded the same way the chunks were, and the vector database returns the `top_k` most similar chunks.

In [8]:
def retrieve_chunks(question, top_k=3):
    """Embed the question and return the top_k most similar chunks with their scores."""
    query_embedding = embedding_model.encode([question], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_embedding, dtype="float32"), top_k)

    results = [
        {"chunk": chunks[idx], "score": float(score)}
        for idx, score in zip(indices[0], scores[0])
    ]
    return results


# Quick sanity check
test_results = retrieve_chunks("What is an autoencoder?", top_k=2)
for r in test_results:
    print(f"score={r['score']:.3f}")
    print(textwrap.fill(r["chunk"], width=100))
    print("-" * 100)

score=0.711
a loss function through gradient descent. Deep Learning Deep learning refers to neural networks with
many hidden layers. These deep architectures can automatically learn increasingly abstract features
from raw data, which is why they perform so well on complex tasks like image recognition, speech
recognition, and natural language processing. Training deep networks generally requires large
amounts of data and significant computing power, often provided by GPUs. Autoencoders An autoencoder
is a type of neural network trained to reconstruct its own input. It consists of an encoder, which
compresses the input into a smaller latent representation, and a decoder, which reconstructs
----------------------------------------------------------------------------------------------------
score=0.617
input. It consists of an encoder, which compresses the input into a smaller latent representation,
and a decoder, which reconstructs the original input from that representation. Autoencoders

# **Answer Generation**

A small instruction-tuned language model (`google/flan-t5-base`) generates the final answer, grounded in the retrieved context. It runs locally on CPU, so no API key is required.

In [9]:
generator = pipeline("text2text-generation", model="google/flan-t5-base")


def generate_answer(question, context, max_new_tokens=150):
    prompt = (
        "Answer the question using only the context below. "
        "If the answer is not in the context, say you don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )
    output = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    return output[0]["generated_text"].strip()

Device set to use cpu


# **Full RAG Pipeline**

Combine retrieval and generation into a single function: retrieve the most relevant chunks, join them into one context block, then generate the answer.

In [10]:
def rag_pipeline(question, top_k=3, verbose=True):
    retrieved = retrieve_chunks(question, top_k=top_k)
    context = "\n\n".join(r["chunk"] for r in retrieved)

    answer = generate_answer(question, context)

    if verbose:
        print(f"Question: {question}\n")
        print("Retrieved context:")
        for i, r in enumerate(retrieved, start=1):
            print(f"  [{i}] (score={r['score']:.3f}) {r['chunk'][:120]}...")
        print(f"\nAnswer: {answer}")
        print("=" * 100)

    return answer, retrieved

# **Test the RAG System**

In [11]:
questions = [
    "What is the main idea of the document?",
    "What is the difference between supervised and unsupervised learning?",
    "What does an autoencoder do?",
    "What problem do LSTMs solve?",
    "What is Retrieval-Augmented Generation?",
]

for q in questions:
    rag_pipeline(q)

Question: What is the main idea of the document?

Retrieved context:
  [1] (score=0.204) and effective at capturing long-range dependencies. Transformers are the foundation of modern large language models, inc...
  [2] (score=0.200) as additional context. This grounds the model's answers in the retrieved information, which reduces hallucination and al...
  [3] (score=0.186) Machine Learning and Deep Learning: Study Notes Introduction to Machine Learning Machine learning (ML) is a branch of ar...

Answer: Machine Learning and Deep Learning: Study Notes Introduction to Machine Learning
Question: What is the difference between supervised and unsupervised learning?

Retrieved context:
  [1] (score=0.629) trained on labeled data, meaning every training example includes both an input and the correct output. Common supervised...
  [2] (score=0.509) Machine Learning and Deep Learning: Study Notes Introduction to Machine Learning Machine learning (ML) is a branch of ar...
  [3] (score=0.433) th

# **Improvements & Experiments**

- Try different chunk sizes / overlaps and measure retrieval quality
- Try a larger or domain-specific embedding model (e.g. `all-mpnet-base-v2`)
- Add hybrid search (combine keyword search like BM25 with vector search)
- Add a re-ranking step on the retrieved chunks before generation
- Swap `flan-t5-base` for a larger local model or a hosted API model (e.g. Claude) for higher-quality answers
- Support real PDFs (`pypdf`) and multiple documents instead of one `.txt` file

# **Conclusion**

This notebook builds an end-to-end RAG pipeline: the source document is chunked, embedded with a sentence-transformer model, and indexed in FAISS for fast similarity search. At query time, the question is embedded and matched against the stored chunks to retrieve the most relevant passages, which are then passed as context to a language model to generate a grounded answer.

Compared to asking a language model a question directly, this approach anchors every answer in the source document, which reduces hallucination and lets the system answer questions about private or domain-specific data the model was never trained on. This same pattern — chunk, embed, store, retrieve, generate — scales from a single text file up to enterprise-scale knowledge assistants and documentation search systems.